In [ ]:
import torch
from torchvision.models.detection import retinanet_resnet50_fpn
from torchvision.ops import sigmoid_focal_loss
from src.detection.model import dinov3_detection
repo_path = "/home/VANDERBILT/zimmejr1/Documents/GitHub/dinov3"
weights_path = "/home/VANDERBILT/zimmejr1/Documents/DinoV3_weights/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth"
model_name  = "dinov3_vitl16"


def _sum(x):
    res = x[0]
    for i in x[1:]:
        res = res + i
    return res
# 1. Define the custom loss logic inside a subclass of the Head
class CalibratedRetinaNetHead(torch.nn.Module):
    def __init__(self, original_head, smoothing, gamma):
        super().__init__()
        self.original_head = original_head
        self.smoothing = smoothing
        self.gamma = gamma

        # This is to fix using det_utils.Matcher.BETWEEN_THRESHOLDS in TorchScript.
        # TorchScript doesn't support class attributes.
        # https://github.com/pytorch/vision/pull/1697#issuecomment-630255584
        self.BETWEEN_THRESHOLDS = -2

    def forward(self, x):
        return self.original_head(x)

    
    def compute_loss(self, targets, head_outputs, matched_idxs):
        losses = []

        cls_logits = head_outputs["cls_logits"]

        for targets_per_image, cls_logits_per_image, matched_idxs_per_image in zip(targets, cls_logits, matched_idxs):
            # determine only the foreground
            foreground_idxs_per_image = matched_idxs_per_image >= 0
            num_foreground = foreground_idxs_per_image.sum()

            # create the target classification
            gt_classes_target = torch.zeros_like(cls_logits_per_image)
            gt_classes_target[
                foreground_idxs_per_image,
                targets_per_image["labels"][matched_idxs_per_image[foreground_idxs_per_image]],
            ] = 1.0

            # --- Apply Label Smoothing ---
            # Formula: target = target * (1 - smoothing) + 0.5 * smoothing
            # This pushes 0.0 to epsilon and 1.0 to 1-epsilon
            if hasattr(self, 'smoothing') and self.smoothing > 0:
                gt_classes_target = gt_classes_target * (1 - self.smoothing) + 0.5 * self.smoothing

            # find indices for which anchors should be ignored
            valid_idxs_per_image = matched_idxs_per_image != self.BETWEEN_THRESHOLDS

            # compute the classification loss
            losses.append(
                sigmoid_focal_loss(
                    cls_logits_per_image[valid_idxs_per_image],
                    gt_classes_target[valid_idxs_per_image],
                    reduction="sum",
                    gamma=self.gamma
                )
                / max(1, num_foreground)
            )

        return _sum(losses) / len(targets)

    # def compute_loss(self, targets, head_outputs, matched_idxs):
    #     total_loss = 0.0
    #     total_num_foreground = 0

    #     cls_logits = head_outputs['cls_logits']

    #     for targets_per_image, cls_logits_per_image, matched_idxs_per_image in zip(targets, cls_logits, matched_idxs):
    #         gt_classes_target = torch.zeros_like(cls_logits_per_image)
    #         foreground_idxs = matched_idxs_per_image >= 0
            
    #         if foreground_idxs.any():
    #             gt_classes_target[foreground_idxs, targets_per_image['labels'][matched_idxs_per_image[foreground_idxs]]] = 1.0

    #         # Apply Label Smoothing
    #         gt_classes_target = gt_classes_target * (1 - self.smoothing) + 0.5 * self.smoothing
            
    #         valid_idxs = matched_idxs_per_image != -1 
            
    #         # Compute loss (keep as sum)
    #         loss = sigmoid_focal_loss(
    #             cls_logits_per_image[valid_idxs], 
    #             gt_classes_target[valid_idxs], 
    #             reduction='sum',
    #             gamma=self.gamma
    #         )
            
    #         total_loss += loss
    #         total_num_foreground += foreground_idxs.sum().item()

    #     # Normalize by the total number of foreground anchors across the batch
    #     return total_loss / max(1, total_num_foreground)
    
# 2. Function to "Patch" your model for Raster Vision
def get_calibrated_dinov3_model(num_classes, smoothing, gamma):
    # This is where you'd initialize your DINOv3 + RetinaNet model
    model = dinov3_detection(
        fine_tune=True,
        num_classes=2, 
        weights=weights_path,
        model_name=model_name,
        repo_dir=repo_path,
        feature_extractor="multi",
        head="retinanet"
    ) 
    
    # Wrap the existing classification head with our calibrated version
    original_head = model.head.classification_head
    model.head.classification_head = CalibratedRetinaNetHead(
        original_head, 
        smoothing=smoothing, 
        gamma=gamma
    )
    return model

In [ ]:
from src.detection.model import dinov3_detection
# repo_path = "/home/VANDERBILT/zimmejr1/Documents/GitHub/dinov3"
# weights_path = "/home/VANDERBILT/zimmejr1/Documents/DinoV3_weights/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth"
# model_name  = "dinov3_vitl16"
# model = dinov3_detection(
#     fine_tune=True,
#     num_classes=2, 
#     weights=weights_path,
#     model_name=model_name,
#     repo_dir=repo_path,
#     feature_extractor="multi",
#     head="retinanet"
# )
model = get_calibrated_dinov3_model(num_classes=2,smoothing=0.01,gamma=2.5)
from rastervision.pytorch_learner.object_detection_utils import TorchVisionODAdapter
model = TorchVisionODAdapter(model)

In [ ]:

# model.model.head.classification_head.loss_function

In [ ]:
import os
import geopandas as gpd

import pathlib

from rastervision.core.data import GeoJSONVectorSource, RasterioCRSTransformer,ClassConfig
from rastervision.pytorch_learner import ClassificationSlidingWindowGeoDataset
from rastervision.core.data import RasterioSource
from geopacha_utilities.utilities import find_pixel_size
TRAINING_AOI_DIRECTORY = '/mnt/sarl_commons06/Wernke_projects/zimmejr1/DinoV3_Chacu_AOI_12_23_25/training_aoi'
VALIDATION_AOI_DIRECTORY = '/mnt/sarl_commons06/Wernke_projects/zimmejr1/DinoV3_Chacu_AOI_12_23_25/validation_aoi_sampled'
LABEL_URI = '/mnt/sarl_commons06/Wernke_projects/zimmejr1/DinoV3_Chacu_AOI_12_23_25/chacu_labels.geojson'
IMAGERY_BASE_DIRECTORY = '/mnt/sarl_commons06/Wernke_projects/GeoPACHA/Imagery_Machine_Learning/Analysis_Images'

patch_dim = 1024

In [ ]:
import albumentations as A

data_augmentation_transform = A.Compose([
    A.D4(p=1.0),
    # A.OneOf([
    #     A.HueSaturationValue(hue_shift_limit=10),
    #     A.RGBShift(),
    #     A.ToGray(),
    #     A.ToSepia(),
    #     A.RandomBrightnessContrast(),
    #     A.RandomGamma(),
    # ]),
    # A.CoarseDropout(max_height=32, max_width=32, max_holes=5)
])

In [ ]:


class_config = ClassConfig(
    names=['chacu', 'background'],
    colors=['darkred', 'gray'],
    null_class='background')



    

In [ ]:
from tqdm import tqdm
from rastervision.pytorch_learner import ObjectDetectionRandomWindowGeoDataset,ObjectDetectionSlidingWindowGeoDataset
from rastervision.pytorch_learner.object_detection_utils import collate_fn as od_collate
from rastervision.core.data import ObjectDetectionLabelSourceConfig
from rastervision.core.data import ObjectDetectionLabelSource

class_config = ClassConfig(
    names=['chacu', 'background'],
    colors=['darkred', 'gray'],
    null_class='background')



training_dataset_list = []
val_dataset_list = []
for aoi_path in tqdm(os.listdir(TRAINING_AOI_DIRECTORY),desc="making datasets"):
        full_aoi_path = os.path.join(TRAINING_AOI_DIRECTORY,aoi_path)
        aoi = gpd.read_file(full_aoi_path)
        image_id = aoi['imageid'][0]
        # print(image_id)
        image_path  = pathlib.PureWindowsPath(aoi['filepath'][0]).as_posix()
        full_image_path = os.path.join(IMAGERY_BASE_DIRECTORY,image_path)

        #Adjust the patch size to account for different resolution
        rasterSource = RasterioSource(
                full_image_path, #path to the image
                allow_streaming=True, # allow_streaming so we don't have to load the whole image
            ) 
        pixel_size = find_pixel_size(rasterSource.imagery_path)
        size = round(patch_dim*.5/pixel_size)



        # print("CREATING DATASET")
        try:
            ds = ObjectDetectionRandomWindowGeoDataset.from_uris(
                image_uri=full_image_path,
                aoi_uri=full_aoi_path,
                label_vector_uri = LABEL_URI,
                class_config=class_config,
                image_raster_source_kw=dict(allow_streaming=True,channel_order=[4,2,1]),
                max_windows=4,
                size_lims = [size,size+1],
                out_size=patch_dim,within_aoi=True,ioa_thresh = 0.75,neg_ratio=.5,
                transform = data_augmentation_transform
            )
            ds.scene.id=image_id
            training_dataset_list.append(ds)
        except:
            try:
                # print("Extracting Negatives")
                ds = ObjectDetectionRandomWindowGeoDataset.from_uris(
                    image_uri=full_image_path,
                    aoi_uri=full_aoi_path,
                    label_vector_uri = LABEL_URI,
                    class_config=class_config,
                    image_raster_source_kw=dict(allow_streaming=True,channel_order=[4,2,1]),
                    max_windows=2,
                    size_lims = [size,size+1],
                    out_size=patch_dim,within_aoi=False,
                    transform = data_augmentation_transform
                )
                ds.scene.id=image_id
                training_dataset_list.append(ds)
            except Exception as e: 
                print(f"Couldn't create dataset because:\n{e}")                   
                continue
print("MAKING VALIDATION DATA")
for aoi_path in tqdm(os.listdir(VALIDATION_AOI_DIRECTORY),desc="making datasets"):
        full_aoi_path = os.path.join(VALIDATION_AOI_DIRECTORY,aoi_path)
        aoi = gpd.read_file(full_aoi_path)
        image_id = aoi['imageid'][0]
        # print(image_id)
        image_path  = pathlib.PureWindowsPath(aoi['filepath'][0]).as_posix()
        full_image_path = os.path.join(IMAGERY_BASE_DIRECTORY,image_path)

        #Adjust the patch size to account for different resolution
        rasterSource = RasterioSource(
                full_image_path, #path to the image
                allow_streaming=True, # allow_streaming so we don't have to load the whole image
            ) 
        pixel_size = find_pixel_size(rasterSource.imagery_path)
        size = round(patch_dim*.5/pixel_size)


        try:
              ds = ObjectDetectionSlidingWindowGeoDataset.from_uris(
                    image_uri = full_image_path,
                    aoi_uri = full_aoi_path,
                    label_vector_uri = LABEL_URI,
                    class_config = class_config,
                    size = size,
                    stride = int(size*0.5),
                    out_size = patch_dim,within_aoi=True,
                    image_raster_source_kw=dict(allow_streaming=True,channel_order=[4,2,1]),return_window=False

              )
              ds.scene.id = image_id
              val_dataset_list.append(ds)
        except Exception as e:
              print(f"Problem with {image_id} sliding window: \n {e}")
              continue

In [ ]:
from torch.utils.data import ConcatDataset
training_data = ConcatDataset(training_dataset_list)
validation_data = ConcatDataset(val_dataset_list)

In [ ]:
# from torch.utils.data import DataLoader
# train_dl = DataLoader(
#     training_data,
#     batch_size=8,
#     # This is the collate function needed for PyTorch object detection models
#     collate_fn=od_collate, 
#     # Must be 0 workers to bypass the multiprocessing error
#     num_workers=0, 
#     shuffle=True
# )

# for batch_idx, (images, targets) in enumerate(train_dl):
#     print(f"Testing Batch {batch_idx + 1}/{len(train_dl)}...")
#     print(len(targets))
#     print(any([len(target.get_field("class_ids")) for target in targets]))

In [ ]:
len(training_data)

In [ ]:
from rastervision.pytorch_learner import ObjectDetectionGeoDataConfig,ObjectDetectionLearnerConfig
from rastervision.pytorch_learner import SolverConfig
data_cfg = ObjectDetectionGeoDataConfig(
    class_config = class_config,
    num_workers=0
)

solver_cfg = SolverConfig(
    batch_sz = 8,
    lr = 5e-5,

    # lr = 1.5*5e-5,
)

learner_cfg = ObjectDetectionLearnerConfig(data=data_cfg,solver=solver_cfg)



In [ ]:
import os

# Set to '0' to use the first GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '0' 

# OR
# Set to '' to use CPU-only (if you want to completely disable GPU access)
# os.environ['CUDA_VISIBLE_DEVICES'] = '' 

print(f"CUDA_VISIBLE_DEVICES is set to: {os.environ.get('CUDA_VISIBLE_DEVICES')}")
from rastervision.pytorch_learner import ObjectDetectionLearner
learner = ObjectDetectionLearner(
    cfg = learner_cfg,
    output_dir = 'data',
    model=model,
    train_ds = training_data,
    valid_ds = validation_data
)

In [ ]:
import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
# Rerun the training after setting this
learner.train(epochs=70)

In [ ]:
learner

In [ ]:
learner.log_data_stats()

In [ ]:
%load_ext tensorboard
%tensorboard --bind_all --logdir "data/train-demo/tb-logs" --reload_interval 10

In [ ]:
import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
# Rerun the training after setting this
# learner.train(epochs=10)

In [ ]:
# Initialize model
from src.detection.model import dinov3_detection
repo_path = "/home/VANDERBILT/zimmejr1/Documents/GitHub/dinov3"
weights_path = "/home/VANDERBILT/zimmejr1/Documents/DinoV3_weights/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth"
model_name  = "dinov3_vitl16"
model = dinov3_detection(
    fine_tune=True,
    num_classes=2, 
    weights=weights_path,
    model_name=model_name,
    repo_dir=repo_path,
    feature_extractor="multi",
    head="retinanet"
)
model = TorchVisionODAdapter(model)

model.load_state_dict(torch.load("data/train-chacu-round3/last-model.pth"))

In [ ]:
(os.listdir(AOI_DIRECTORY))[0]

In [ ]:
from rastervision.core.data import ObjectDetectionLabels
AOI_DIRECTORY = "/home/VANDERBILT/zimmejr1/Documents/GitHub/dinov3-james-experiments/OD_data/Buffered_Chacu_AOI/ExpandedSecondPass"
# val_dataset_list = []
for aoi_path in [(os.listdir(AOI_DIRECTORY))[0]]:
      print("Making dataset")
      full_aoi_path = os.path.join(AOI_DIRECTORY,aoi_path)
      aoi = gpd.read_file(full_aoi_path)
      image_id_aoi = aoi['imageid'][0]
      image_id = image_id_aoi.split('_')[0]
      output_path = os.path.join('/home/VANDERBILT/zimmejr1/Documents/GitHub/dinov3_stack/data/train-chacu-round3/predictions2',f"{image_id}_chacu_labels.geojson")
      print(image_id)

      if os.path.exists(output_path): continue
      image_path  = pathlib.PureWindowsPath(aoi['filepath'][0]).as_posix()
      full_image_path = os.path.join(IMAGERY_BASE_DIRECTORY,image_path)

      try:
            #Adjust the patch size to account for different resolution
        rasterSource = RasterioSource(
        full_image_path, #path to the image
        allow_streaming=True, # allow_streaming so we don't have to load the whole image
        ) 
        pixel_size = find_pixel_size(rasterSource.imagery_path)
        size = round(patch_dim*.5/pixel_size)
        ds = ObjectDetectionSlidingWindowGeoDataset.from_uris(
              image_uri = full_image_path,
              # aoi_uri = full_aoi_path,
              # label_vector_uri = LABEL_URI,
              class_config = class_config,
              size = size,
              stride = int(size/2),
              out_size = patch_dim,
              image_raster_source_kw=dict(allow_streaming=True,channel_order=[4,2,1]),return_window=False

              )
        ds.scene.id = image_id
        predictions = learner.predict_dataset(
          ds,
          raw_out=True,
          numpy_out=True,
          progress_bar=True,
          dataloader_kw = {'num_workers':16,'batch_size':8})

        pred_labels = ObjectDetectionLabels.from_predictions(
          ds.windows,
          predictions,
          )
        pred_labels_2 = pred_labels.prune_duplicates(pred_labels,score_thresh=.1,merge_thresh=.1)
        pred_labels_2.save(output_path,class_config=class_config,crs_transformer=ds.scene.raster_source.crs_transformer)
      except Exception as e:
          print(f"Skipping {image_id} because of : {e} ")
          continue
          


In [ ]:
learner.save_model_bundle()

In [ ]:

for ds in val_dataset_list:

    ds = val_dataset_list[2]
    

In [ ]:
from rastervision.core.data import ObjectDetectionLabels

ds = val_dataset_list[2]
predictions = learner.predict_dataset(
    ds,
    raw_out=True,
    numpy_out=True,
    progress_bar=True)

pred_labels = ObjectDetectionLabels.from_predictions(
    ds.windows,
    predictions,
    )

In [ ]:
pred_labels.save("test_pred_labels.geojson",class_config=class_config,crs_transformer=ds.scene.raster_source.crs_transformer)

In [ ]:
pred_labels_2 = pred_labels.prune_duplicates(pred_labels,score_thresh=.3,merge_thresh=.8)

In [ ]:
pred_labels_2.save("test_pred_labels2.geojson",class_config=class_config,crs_transformer=ds.scene.raster_source.crs_transformer)

In [ ]:
test_labels= ObjectDetectionLabels.from_predictions(validation_data.datasets[0].windows,test)

In [ ]:
test_labels.get_boxes()

In [ ]:
valid_dl = learner.get_dataloader('valid')
preds = learner.predict_dataloader(
            valid_dl, return_format='xyz', batched_output=True, raw_out=True)

In [ ]:
x,y,z = next(preds)

In [ ]:
x.shape

In [ ]:
from os.path import join

import matplotlib.pyplot as plt
import torch

from rastervision.pytorch_learner.dataset import (
    SemanticSegmentationSlidingWindowGeoDataset,
    ObjectDetectionSlidingWindowGeoDataset,
    ClassificationSlidingWindowGeoDataset)
from rastervision.pytorch_learner.dataset.visualizer import (
    SemanticSegmentationVisualizer,
    ObjectDetectionVisualizer,
    ClassificationVisualizer)
from rastervision.core.data import ClassConfig
# channel_display_groups = {'RGB': (4, 2, 1)}
vis = ObjectDetectionVisualizer(
    class_names=class_config.names, class_colors=class_config.colors)

filtered_outputs = [img_pred.score_filter(score_thresh=.3) for img_pred in z]
vis.plot_batch(x,z=filtered_outputs, show=True)

In [ ]:
for img_pred in z:
    img_pred.score_filter()

In [ ]:
model

In [ ]:
import rastervision.pytorch_learner as rv_plt
print([item for item in dir(rv_plt) if 'Dataset' in item])

In [ ]:
from rastervision.pytorch_learner import (
    ObjectDetectionGeoDataConfig, 
    ObjectDetectionLearnerConfig,
)

# 1. Initialize the GeoDatasetConfig (can be empty for inference)

# 2. Update your data_cfg
data_cfg = ObjectDetectionGeoDataConfig(
    class_config=class_config,
    train_scenes=[],# This satisfies the internal check
    num_workers=0
)

# 3. Recreate the learner config
learner_cfg = ObjectDetectionLearnerConfig(data=data_cfg, solver=solver_cfg)

# 4. Initialize Learner and load weights
learner = ObjectDetectionLearner(
    cfg=learner_cfg,
    output_dir='data',
    model=model 
)
learner.load_state_dict('data/model.pth')

In [ ]:
from rastervision.pytorch_learner import ObjectDetectionLearner
os.environ['CUDA_VISIBLE_DEVICES'] = '0' 



# 2. Instantiate the learner
# Note: 'model' should be the same architecture instance used before
learner = ObjectDetectionLearner(
    cfg=learner_cfg,
    output_dir='data',
    model=model 
)

# 3. Load the saved weights
# Replace 'model.pth' with your actual filename
learner.load_state_dict('data/model.pth')

In [ ]:
learner.plot_predictions(split='valid', show=True)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. Load your spreadsheet
df = pd.read_csv('/home/VANDERBILT/zimmejr1/Desktop/reliability_plot_csv_partial.csv') 
num_bins=11
# 2. Define the bins (e.g., 10 bins from 0 to 1)
bins = np.linspace(0.5, 1, num_bins)
df['bin'] = pd.cut(df['score'], bins=bins, include_lowest=True)

# 3. Calculate accuracy (mean of is_chacu) per bin
# Since is_chacu is boolean, its mean is the frequency of True values
calibration = df.groupby('bin')['is_chacu'].mean().reset_index()

# 4. Get the center of each bin for the x-axis
calibration['bin_center'] = calibration['bin'].apply(lambda x: x.mid)

bar_width = (0.5 / num_bins) * 0.9
# 5. Plot the results
plt.figure(figsize=(10, 6))
plt.bar(calibration['bin_center'], calibration['is_chacu'], width=bar_width, alpha=0.7, edgecolor='black')

# Add a diagonal line representing "Perfect Calibration"
plt.plot([0.5, 1], [0.5, 1], linestyle='--', color='red', label='Perfectly Calibrated')

plt.xlabel('Model Confidence Score')
plt.ylabel('Accuracy (Frequency of Correct Prediction)')
plt.title('Model Calibration Plot')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

In [ ]:
import pandas as pd
import numpy as np

def calculate_ece(df, score_col='score', target_col='is_chacu', n_bins=10):
    # 1. Convert boolean to 1/0 if necessary
    y_true = df[target_col].astype(int).values
    y_prob = df[score_col].values
    
    # 2. Define bins
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    
    ece = 0.0
    
    # 3. Calculate error for each bin
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        # Find samples that fall into this confidence bin
        in_bin = (y_prob > bin_lower) & (y_prob <= bin_upper)
        prop_in_bin = np.mean(in_bin) # This is |Bm| / n
        
        if prop_in_bin > 0:
            # Actual accuracy in this bin
            accuracy_in_bin = np.mean(y_true[in_bin])
            # Average confidence in this bin
            avg_confidence_in_bin = np.mean(y_prob[in_bin])
            
            # Weighted absolute difference
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
            
    return ece

# Usage:
# df = pd.read_csv("your_file.csv") # or pd.read_excel(...)
ece_value = calculate_ece(df, score_col='score', target_col='is_chacu')

print(f"Expected Calibration Error: {ece_value:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.isotonic import IsotonicRegression

# 1. Load your data
# df = pd.read_csv("your_data.csv")

# 2. Prepare data (Isotonic Regression needs X as scores and y as 0/1)
X = df['score'].values
y = df['is_chacu'].astype(int).values

# 3. Fit the calibrator
# This learns the mapping to fix that 0.17 error
iso_reg = IsotonicRegression(out_of_bounds='clip')
iso_reg.fit(X, y)

# 4. Generate new, calibrated scores
df['calibrated_score'] = iso_reg.predict(X)

# Now, if you recalculate ECE on 'calibrated_score', 
# it should be significantly lower (likely < 0.05).
df.to_csv("calibrated_results.csv", index=False)

In [ ]:
calculate_ece(df,score_col='calibrated_score',target_col='is_chacu')